# 💧 AquaSense AI — 05: SHAP Interpretability Analysis
**Project:** Intelligent Water Quality Assessment and Potability Prediction Using Explainable Machine Learning  

### Overview:
In this notebook, we apply **SHAP (SHapley Additive exPlanations)** to compute game-theoretic feature attributions for our top model.
We analyze:
- Global Beeswarm summary plot
- Global mean absolute SHAP bar chart
- Local Waterfall plots across 3 distinct test cases (High confidence potable, High confidence non-potable, Uncertain/boundary)
- Feature dependence & interaction plots


In [ ]:
import sys
import os
sys.path.append(os.path.abspath('..'))

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from src.data.loader import DataLoader
from src.data.preprocessor import WaterQualityPreprocessor
from src.models.trainer import ModelTrainer
from src.explainability.shap_explainer import SHAPExplainer
from src.utils.visualization import set_plot_style

set_plot_style()
print("SHAP modules loaded.")


## 1. Load Data, Preprocessor & Best Model


In [ ]:
dl = DataLoader(data_path="../data/raw/water_potability.csv")
df = dl.load()
X_train, X_test, y_train, y_test = dl.split(df, test_size=0.20, random_state=42)

preprocessor = WaterQualityPreprocessor.load("../models/preprocessor.pkl")
X_train_proc = preprocessor.transform(X_train)
X_test_proc = preprocessor.transform(X_test)

best_model = ModelTrainer.load_single("best_model", path="../models")
print("Model loaded:", type(best_model))


## 2. Fit SHAP Explainer & Compute Global Values


In [ ]:
shap_explainer = SHAPExplainer()
shap_explainer.fit(best_model, X_train_proc)
shap_explanation = shap_explainer.global_explanation(X_test_proc)
print("SHAP computation complete.")


## 3. Global Beeswarm Summary Plot


In [ ]:
beeswarm_fig = shap_explainer.plot_beeswarm(shap_explanation, X_test_proc, max_display=12)
plt.show()


## 4. Mean Absolute SHAP Importance Bar Chart


In [ ]:
bar_fig = shap_explainer.plot_bar(shap_explanation, max_display=12)
plt.show()

ranking = shap_explainer.get_feature_ranking(shap_explanation, preprocessor.feature_names_out_)
pd.DataFrame(ranking, columns=['Feature', 'Mean |SHAP Value|']).head(10)


## 5. Local Waterfall Plots (3 Representative Test Cases)


In [ ]:
# Case 1: High confidence potable
# Case 2: High confidence not potable
# Case 3: Uncertain / boundary sample
probs = best_model.predict_proba(X_test_proc)[:, 1]

idx_potable = int(np.argmax(probs))
idx_not_potable = int(np.argmin(probs))
idx_uncertain = int(np.argmin(np.abs(probs - 0.5)))

print(f"Sample #{idx_potable} Prob: {probs[idx_potable]:.3f} (High Potable)")
fig1 = shap_explainer.plot_waterfall(shap_explanation, idx=idx_potable)
plt.show()

print(f"Sample #{idx_not_potable} Prob: {probs[idx_not_potable]:.3f} (High Not Potable)")
fig2 = shap_explainer.plot_waterfall(shap_explanation, idx=idx_not_potable)
plt.show()

print(f"Sample #{idx_uncertain} Prob: {probs[idx_uncertain]:.3f} (Uncertain)")
fig3 = shap_explainer.plot_waterfall(shap_explanation, idx=idx_uncertain)
plt.show()


## 6. Feature Dependence Plots


In [ ]:
fig_dep = shap_explainer.plot_dependence(shap_explanation, X_test_proc, feature='Sulfate')
plt.show()
